In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
from pathlib import Path
# Load the dataset
Project_Root = Path.cwd().parent

Data_Path = Project_Root / "Data"/"processed"/"Gurgaon_merged_v2.csv"
df = pd.read_csv(Data_Path)
df.sample(5)


FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\Admin\\Data\\processed\\Gurgaon_merged_v2.csv'

In [ ]:
# Fill missing values in 'built_up_area' with corresponding values from 'Plot_area'
df['built_up_area'] = df['built_up_area'].fillna(df['Plot_area'])

In [ ]:
df.drop(columns = "Plot_area", inplace = True)

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.duplicated().sum() # Check of duplicates have been removed

## Univariate Analysis with Property type

In [ ]:
df.property_type.value_counts().plot(kind = 'bar', figsize = (10,5), title = "Property Type Distribution", legend = True, )

# Observation
The Data consists of 70 to 75 percent of the property type and remaining house.

# READ: property_type ~70-75% flats, rest houses. Imbalanced but flats dominate — most models will key off this.


# Society

In [ ]:
df.sample(1)

In [ ]:
df.society.value_counts().shape

In [ ]:
df.society.value_counts()

In [ ]:
df[df['society'] != "independent"]['society'].value_counts(normalize=True).cumsum().head(75)

In [ ]:
society_counts = df.society.value_counts()

# Frequency distribution of the society column
frequency_bin = {
    'High(>50)'        : (society_counts>50).sum(),
    'Average(10 - 49)' : ((society_counts>10) & (society_counts<50)).sum(),
    'Low(2 - 9)'       : ((society_counts>2) & (society_counts<10)).sum(),
    'very Low (1)'     : (society_counts ==1).sum()
}
frequency_bin

In [ ]:
df.society.unique().shape

In [ ]:
df.society.value_counts(normalize = True) * 100

# observations
--> There are 14 percent independent societies.
--> There are 724 unique societies.
--> The top 75 societies has 50 percent of the properties.

# READ: society — 724 unique, heavy long tail. 14% independent. Top 75 societies = 50% of listings. High cardinality → bucket to top-N + 'Other'.


In [ ]:
df


## Sector

In [ ]:
df.sector.value_counts().shape

In [ ]:
df.sector.isnull().sum()

In [ ]:
df.sector.value_counts().head(10).plot(kind= 'bar')

In [ ]:
# Frequency distribution of the sector column
sector_counts = df.sector.value_counts()
frequency_bin_sector = {
    'Very High(>100)'        : (sector_counts>100).sum(),
    'High(50 - 100)' : ((sector_counts>=50) & (sector_counts<=100)).sum(),
    'Average(10 - 49)' : ((sector_counts>=10) & (sector_counts<=49)).sum(),
    'Low(2 - 9)'       : ((sector_counts>=2) & (sector_counts<=9)).sum(),
    'Very Low (1)'     : (sector_counts ==1).sum()
}
frequency_bin_sector

# Observation
There are 156 unique sectors.

# READ: sector — 156 unique, a few dense sectors + long tail of sparse ones. Sparse sectors give unreliable per-sector price stats.


# Price

In [ ]:
df.price.isnull().sum()

In [ ]:
df.price.describe()

In [ ]:
# Outlier detection using IQR method
IQR = df.price.quantile(0.75) - df.price.quantile(0.25)
lower_bound = df.price.quantile(0.25) - 1.5 * IQR
upper_bound = df.price.quantile(0.75) + 1.5 * IQR
print(f"Lower Bound: {lower_bound}, Upper Bound: {upper_bound}")
Outliers = df[(df.price < lower_bound) | (df.price > upper_bound)]
print(f"Number of outliers: {(df.price < lower_bound).sum() + (df.price > upper_bound).sum()}")

In [ ]:
Outliers.head()

In [ ]:
Outliers.price.describe()

In [ ]:
sns.histplot(df.price, bins=30, kde=True)
plt.title("Distribution of House Prices")
plt.xlabel("Price")
plt.ylabel("Frequency")
plt.show()


In [ ]:
sns.boxplot(x=df.price, color = 'skyblue')
plt.title("Boxplot of House Prices")
plt.grid()

In [ ]:
# Skewness and Kurtosis
skewness = df['price'].skew()
kurtosis = df['price'].kurt()

print(skewness,kurtosis)

# READ: price — strongly right-skewed (skew >0, heavy kurtosis). Mean >> median. Long luxury tail; IQR flags many UPPER outliers (real premium homes, not errors).


In [ ]:
# Price Binning
bins = [0,1,2,3,5,8,10,15,20,25]
bins_label = [" 0-1", " 1-2", " 2-3", " 3-5", " 5-8", " 8-10", " 10-15", " 15-20", " 20-25"]
pd.cut(df['price'], bins=bins, labels=bins_label).value_counts().sort_index().plot(kind='bar', figsize=(10,5), title="Price Binning Distribution", legend=True)

In [ ]:
# ecdf plot
ecdf = df.price.value_counts().sort_index().cumsum() / len(df['price'])
plt.plot(ecdf.index, ecdf, marker='.', linestyle='none')
plt.xlabel('Price')
plt.ylabel('ECDF')
plt.title('Empirical Cumulative Distribution Function (ECDF) of House Prices')
plt.show()

In [ ]:
# Applying log transformation on the price column as to see the difference
plt.figure(figsize=(15, 6))
plt.subplot(1, 2, 1)
sns.histplot(df.price, kde=True, bins=50, color='skyblue')
plt.title("Distribution of House Prices (Original)")
plt.xlabel("Price")
plt.ylabel("Frequency")

plt.subplot(1, 2, 2)
sns.histplot(np.log1p(df.price), kde=True, bins=50, color='orange')
plt.title("Distribution of House Prices (Log Transformed)")
plt.xlabel("Log(Price)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
skewness = np.log1p(df['price']).skew()
kurtosis = np.log1p(df['price']).kurt()

print(skewness,kurtosis)

# READ: log(price) — skew & kurtosis collapse toward 0, near-symmetric. Model on log-price, back-transform predictions.


In [ ]:
plt.figure(figsize=(15, 6))

# Distribution plot without log transformation
plt.subplot(1, 2, 1)
sns.boxplot(df['price'], color='skyblue')
plt.title('Distribution of Prices (Original)')
plt.xlabel('Price (in Crores)')
plt.ylabel('Frequency')

# Distribution plot with log transformation
plt.subplot(1, 2, 2)
sns.boxplot(np.log1p(df['price']), color='lightgreen')
plt.title('Distribution of Prices (Log Transformed)')
plt.xlabel('Log(Price)')
plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

## Price_sq_feet

In [ ]:
df.price_per_sqft.isnull().sum()

In [ ]:
df.price_per_sqft.describe()

In [ ]:
sns.histplot(df.price_per_sqft, bins=30, kde=True)
plt.title("Distribution of House Prices")
plt.xlabel("Price (in Crores)")
plt.ylabel("Frequency")
plt.show()

In [ ]:
sns.boxplot(x=df.price_per_sqft, color = 'skyblue')
plt.show()

# READ: price_per_sqft — right-skewed. Extreme highs are likely upstream area/price errors, not true luxury. Use as a data-quality tripwire.


# Bedroom

In [ ]:
df.bedRoom.value_counts().plot(kind = 'bar', figsize = (10,5), title = "Bedroom Distribution", legend = True)
plt.xlabel("Number of Bedrooms")
plt.ylabel("Frequency")
plt.show()

In [ ]:
df.bedRoom.value_counts(normalize = True).head(5).plot(kind = 'pie', autopct='%1.1f%%', figsize = (10,5), title = "Bedroom Distribution", legend = True)

# READ: bedRoom — 2/3/4 BHK dominate. Very high bedroom counts (tiny freq) are suspect data-entry errors.


# Bathroom

In [ ]:
df.bathroom.value_counts()

In [ ]:
df.bathroom.isnull().sum()

In [ ]:
df.bathroom.value_counts().sort_index().plot(kind = 'bar')

In [ ]:
df.bathroom.value_counts(normalize = True).head().plot(kind = 'pie', autopct = '%0.2f%%')

# READ: bathroom — tracks bedrooms closely. Note for multicollinearity at modelling time.


In [ ]:
df.balcony.value_counts()

In [ ]:
df.balcony.value_counts().sort_index().plot(kind = 'bar')

In [ ]:
df.balcony.value_counts(normalize = True).plot(kind = 'pie', autopct = '% 0.2f%%')

# READ: balcony — concentrated on low counts; treat as categorical (incl. '3+'/'No' style level).


# FLoor_num


In [ ]:
df.head()

In [ ]:
df.floorNum.isnull().sum()

In [ ]:
df.floorNum.value_counts().sort_index().plot(kind = 'bar', figsize = (10,5), title = "Floor Number Distribution", legend = True)

In [ ]:
df[df.floorNum.isnull()]

In [ ]:
df.floorNum.describe()

In [ ]:
sns.boxplot(df.floorNum,  color = "lightgreen")

# READ: floorNum — concentrated on low/mid floors. Some nulls + extreme highs to verify against real building stock.


In [ ]:
df.facing.isnull().sum()

In [ ]:
df.facing.fillna("NA", inplace = True)
df.facing.value_counts()

# READ: facing — heavy missingness, filled as explicit 'NA' category. Absence may itself be signal.


In [ ]:
df.agePossession.isnull().sum()

In [ ]:
df.agePossession.value_counts()

In [ ]:
df['agePossession'] = df['agePossession'].replace('undefined', 'Undefined')

In [ ]:
df.agePossession.value_counts()

# READ: agePossession — messy labels cleaned ('undefined'→'Undefined'). Mostly ready-to-move vs under-construction tiers.


## Areas

In [ ]:
df.head(2)

In [ ]:
df.super_built_up_area.isnull().sum()

In [ ]:
df.super_built_up_area.describe()

In [ ]:
sns.histplot(df.super_built_up_area, bins=30, kde=True)

In [ ]:
sns.boxplot(df.super_built_up_area, color = 'lightgreen')

# READ: super_built_up_area — right-skewed, heavily missing (rarely co-present with the other area cols).


In [ ]:
df.built_up_area.isnull().sum()

In [ ]:
df.built_up_area.describe()

In [ ]:
sns.histplot(df.built_up_area, bins=30, kde=True)

In [ ]:
sns.boxplot(df.built_up_area.dropna(), color = 'lightgreen')

Lot more do with this column as extreme outliers detected.

# READ: built_up_area — extreme outliers, largely a side-effect of the earlier Plot_area fill (plot >> built-up). Not real signal.


In [ ]:
df.carpet_area.isnull().sum()

In [ ]:
df.carpet_area.describe()

In [ ]:
sns.histplot(df.carpet_area, bins=30, kde=True)

In [ ]:
sns.boxplot(df.carpet_area, color = 'lightgreen')

Need to work with the outliers here.

# READ: carpet_area — right-skewed with outliers. Across all 3 area cols: reconcile into ONE feature (collinear + inconsistently populated).


# Additional Rooms.

In [ ]:
df.head()

In [ ]:
plt.figure(figsize = (20,12))

# creating a subplot for each room type
for idx, room_type in enumerate(['study room', 'servant room', 'store room','pooja room','others'], 1):
    ax = plt.subplot(2, 3, idx)
    df[room_type].value_counts().plot.pie(autopct='%1.1f%%', startangle=90, shadow=True, legend=True, title=f"{room_type} Distribution")

plt.tight_layout()
plt.show()

# READ: extra rooms (study/servant/store/pooja/others) — low-prevalence binary flags. Weak alone; combine into a 'premium amenities' count.


In [ ]:
# Furnishing status distribution
df.furnishing_type.value_counts().plot(kind = 'bar', figsize = (10,5), title = "Furnishing Type Distribution", legend = True)

In [ ]:
df.furnishing_type.value_counts(normalize = True).plot(kind = 'pie', autopct = '% 0.2f%%', figsize = (10,5), title = "Furnishing Type Distribution", legend = True)

# READ: furnishing_type — unfurnished/semi/fully. Plausible price driver, keep it.


In [ ]:
# Luxury status distribution
df.luxury_score.value_counts().sort_index()

In [ ]:
df.luxury_score.describe()

In [ ]:
sns.histplot(df.luxury_score, bins=30, kde=True)

In [ ]:
sns.boxplot(df.luxury_score, color = 'lightgreen')

Multimodal


no outliers.

# READ: luxury_score — MULTIMODAL (distinct property tiers), well-bounded, NO true outliers. Consider binning into tiers vs treating as smooth-continuous.
